In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits     # small 8x8 handwritten-digit dataset

# Step 1: Load and normalise the handwritten digits (1797 images, 8 x 8 pixels)
digits = load_digits()
X = digits.data / 16.0
y = digits.target
idx = np.random.RandomState(3).permutation(len(X))
X, y = X[idx], y[idx]
X_train, y_train = X[:1400], y[:1400]
X_test, y_test = X[1400:], y[1400:]

# Step 2: Same network and same initial weights for both optimizers
def build_model():
    tf.keras.utils.set_random_seed(3)
    return keras.Sequential([
        keras.layers.Input(shape=(64,)),
        keras.layers.Dense(32, activation="relu"),
        keras.layers.Dense(10, activation="softmax")])

optimizers = {
    "SGD": keras.optimizers.SGD(learning_rate=0.01),
    "SGD + Momentum (0.9)": keras.optimizers.SGD(learning_rate=0.01, momentum=0.9),
}
epochs = 30
losses = {}

for name, opt in optimizers.items():
    model = build_model()
    model.compile(optimizer=opt, loss="sparse_categorical_crossentropy",
                  metrics=["accuracy"])
    h = model.fit(X_train, y_train, epochs=epochs, batch_size=32, verbose=0)
    losses[name] = np.array(h.history["loss"])
    test_acc = model.evaluate(X_test, y_test, verbose=0)[1]
    print(f"{name:22s} final training loss = {losses[name][-1]:.4f}, "
          f"test accuracy = {test_acc:.4f}")

# Step 3: Compare epoch-wise losses using NumPy
sgd_loss, mom_loss = losses["SGD"], losses["SGD + Momentum (0.9)"]
print("\nEpoch   SGD loss   Momentum loss")
for e in [1, 5, 10, 20, 30]:
    print(f"{e:5d}   {sgd_loss[e-1]:8.4f}   {mom_loss[e-1]:10.4f}")
target = 0.10
for name, v in losses.items():
    r = np.where(v < target)[0]
    print(f"{name:22s} epochs to reach loss < {target}: ",
          int(r[0]) + 1 if len(r) else "not reached in 30 epochs")

# Step 4: Plot the convergence curves
plt.figure(figsize=(8, 5))
plt.plot(range(1, epochs + 1), sgd_loss, "b-o", markersize=3, label="SGD")
plt.plot(range(1, epochs + 1), mom_loss, "r-o", markersize=3, label="SGD + Momentum")
plt.xlabel("Epoch")
plt.ylabel("Training loss (cross-entropy)")
plt.title("SGD vs Momentum: Handwritten Digit Classification")
plt.legend()
plt.grid(True)
plt.show()

SGD                    final training loss = 0.5489, test accuracy = 0.9043
